# MAE задержки: размеченный test и скрытый validate
Test используется для выбора числа деревьев. Затем CatBoost обучается заново на train + test и сохраняет `submission.csv` для validate. Скрытую MAE validate можно узнать только после проверки на платформе.

In [ ]:
DRIVE_DATASET_ZIP = '/content/drive/MyDrive/dataset.zip'
DRIVE_RESULTS_DIR = '/content/drive/MyDrive/mos-trans/mae-final'
BRANCH = 'ya-dolbayob'


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from shutil import copy2
import hashlib, os, subprocess, sys
source_zip = Path(DRIVE_DATASET_ZIP)
if not source_zip.is_file():
    raise FileNotFoundError(source_zip)
local_zip = Path('/content/dataset.zip')
copy2(source_zip, local_zip)
with local_zip.open('rb') as stream:
    digest = hashlib.file_digest(stream, 'sha256').hexdigest()[:16]
repo = Path('/content/Mos-TRANS')
if not repo.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', 'https://github.com/epitaph76/Mos-TRANS.git', str(repo)], check=True)
else:
    subprocess.run(['git', '-C', str(repo), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(repo), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only', 'origin', BRANCH], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(repo / 'requirements-catboost.txt')], check=True)
os.chdir(repo)
revision = subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip()
print('Dataset hash:', digest, 'commit:', revision)


In [ ]:
from mos_trans.preprocessing import build_dataset
from mos_trans.modeling.catboost import run
processed_dir = Path('/content/processed')
if not (processed_dir / 'train_features.parquet').is_file():
    build_dataset(local_zip, processed_dir)
result_dir = Path(DRIVE_RESULTS_DIR) / f'{digest}-{revision}'
report = run(processed_dir, result_dir, local_zip)
print('Test MAE (test used for early stopping):', report['catboost']['mae_delay_s'])
print('Final training rows:', report['final_model_training_rows'])
print('Submission:', result_dir / 'submission.csv')


In [ ]:
import pandas as pd
submission = pd.read_csv(result_dir / 'submission.csv', sep=';')
assert submission.columns.tolist() == ['sample_id', 'prediction']
assert len(submission) == submission.sample_id.nunique() == 151
assert submission.prediction.notna().all()
display(submission.head())
